In [1]:
import re
from pyspark.sql import SparkSession, functions as F, Window


POSTS_PATH = "file:///root/posts_sample.xml"
LANGUAGES_PATH = "file:///root/programming-languages.csv"
OUTPUT_PATH = "file:///root/output/top_languages_2010_2020.parquet"


def normalize_language_to_tag(name: str) -> str | None:
    if name is None:
        return None

    s = name.strip().lower()
    if not s:
        return None

    s = re.sub(r"\s*\(.*?\)\s*", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    aliases = {
        "visual basic .net": "vb.net",
        "visual basic": "visual-basic",
        "objective c": "objective-c",
        "common lisp": "common-lisp",
        "assembly language": "assembly",
        "emacs lisp": "elisp",
        "wolfram language": "mathematica",
        "go!": "go",
    }

    if s in aliases:
        return aliases[s]
        
    return s.replace(" ", "-")


def main():
    spark = (
        SparkSession.builder
        .appName("Top programming languages 2010-2020")
        .config("spark.sql.session.timeZone", "UTC")
        .getOrCreate()
    )

    # -----------------------------
    # 1. Читаем posts XML
    # -----------------------------
    posts = (
        spark.read
        .format("xml")
        .option("rowTag", "row")
        .load(POSTS_PATH)
        .select(
            F.col("_Id").alias("post_id"),
            F.col("_PostTypeId").cast("int").alias("post_type_id"),
            F.col("_CreationDate").alias("creation_date"),
            F.col("_Tags").alias("tags")
        )
    )

    # -----------------------------
    # 2. Читаем CSV со списком языков
    # -----------------------------
    languages_raw = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .csv(LANGUAGES_PATH)
    )

    possible_cols = [c for c in languages_raw.columns if c.lower() in {"name", "language", "programming_language"}]
    language_col = possible_cols[0] if possible_cols else languages_raw.columns[0]

    language_names = [
        row[0] for row in languages_raw.select(F.col(language_col)).distinct().collect()
        if row[0] is not None
    ]

    tag_to_language = {}
    for lang in language_names:
        tag = normalize_language_to_tag(lang)
        if tag and tag not in tag_to_language:
            tag_to_language[tag] = lang.strip()

    languages_prepared = list(tag_to_language.items())
    # [(tag, language_display), ...]

    languages_df = spark.createDataFrame(
        [(tag, display) for tag, display in languages_prepared],
        ["tag", "language"]
    )

    # -----------------------------
    # 3. Берём только вопросы 2010-2020
    # -----------------------------
    questions = (
        posts
        .filter(F.col("post_type_id") == 1)
        .filter(F.col("tags").isNotNull())
        .filter(F.col("creation_date").isNotNull())
        .withColumn("creation_ts", F.to_timestamp("creation_date"))
        .withColumn("year", F.year("creation_ts"))
        .filter((F.col("year") >= 2010) & (F.col("year") <= 2020))
    )

    # -----------------------------
    # 4. Разбираем строку тегов вида <java><spring><hibernate>
    # -----------------------------
    question_tags = (
        questions
        .withColumn(
            "tag_array",
            F.split(
                F.regexp_replace(
                    F.regexp_replace(F.col("tags"), r"^<|>$", ""),
                    r"><",
                    "|"
                ),
                r"\|"
            )
        )
        .select(
            "post_id",
            "year",
            F.explode("tag_array").alias("tag")
        )
        .withColumn("tag", F.lower(F.col("tag")))
    )

    language_questions = (
        question_tags
        .join(F.broadcast(languages_df), on="tag", how="inner")
    )

    # -----------------------------
    # 5. Считаем количество вопросов по (год, язык)
    # -----------------------------
    yearly_counts = (
        language_questions
        .groupBy("year", "language")
        .agg(F.countDistinct("post_id").alias("question_count"))
    )

    # -----------------------------
    # 6. Строим top-10 по каждому году
    # -----------------------------
    w = Window.partitionBy("year").orderBy(
        F.desc("question_count"),
        F.asc("language")
    )

    result = (
        yearly_counts
        .withColumn("rank", F.row_number().over(w))
        .filter(F.col("rank") <= 10)
        .select("year", "rank", "language", "question_count")
        .orderBy("year", "rank")
    )

    result.show(200, truncate=False)

    # -----------------------------
    # 7. Сохраняем в Apache Parquet
    # -----------------------------
    (
        result
        .write
        .mode("overwrite")
        .parquet(OUTPUT_PATH)
    )

    spark.stop()


if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 11:12:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+----+----+-----------------------------------+--------------+
|year|rank|language                           |question_count|
+----+----+-----------------------------------+--------------+
|2010|1   |Java                               |52            |
|2010|2   |PHP                                |46            |
|2010|3   |JavaScript                         |44            |
|2010|4   |Python                             |26            |
|2010|5   |Objective-C                        |23            |
|2010|6   |C                                  |20            |
|2010|7   |Ruby                               |12            |
|2010|8   |Delphi                             |8             |
|2010|9   |AppleScript                        |3             |
|2010|10  |Bash                               |3             |
|2011|1   |PHP                                |102           |
|2011|2   |Java                               |93            |
|2011|3   |JavaScript                         |83      